# IncidentCommander RL — Kaggle shard 1 / 3

**Workload:** every 3rd task starting at index **0** (~127 of 381 scenarios).
Trains a LoRA on **Phi-3.5-mini-instruct** using a local **DeepSeek-R1-0528-Qwen3-8B** critic.

## REQUIRED — attach these 2 Kaggle Models before running
Right sidebar → `+ Add Input` → `Models` tab:
1. `Microsoft / phi-3` → framework `PyTorch` → variation `phi-3.5-mini-instruct` → version `2`
2. `deepseek-ai / deepseek-r1-0528` → framework `Transformers` → variation `deepseek-r1-0528-qwen3-8b` → version `1`

## Required notebook settings
- Accelerator: `GPU T4 x2` or `GPU P100`
- Internet: `On`
- Persistence: `Files only`

**Output:** `/kaggle/working/adapter_kaggle1.zip` — download from the sidebar after the run finishes.

In [ ]:
# === 1. Install deps + unsloth (best-effort) ===
# Qwen3 (in DeepSeek-R1-0528) needs transformers >= 4.51. Unsloth speeds up
# the actor ~2x; if its install fails on this Kaggle image we fall back to
# pure HF transformers automatically (train_lib.py handles both paths).
import subprocess, sys

def pip(*args):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args],
                          check=False).returncode

rc = pip('-U', 'unsloth')
print('[install] unsloth rc =', rc, '(non-zero is fine, HF fallback works)')

pip('-U',
    'transformers>=4.51,<4.55',
    'peft>=0.13,<0.16',
    'accelerate>=1.1,<1.5',
    'bitsandbytes>=0.45.5',
    'huggingface_hub>=0.25,<1.0',
    'pydantic>=2,<3',
    'datasets', 'sentencepiece', 'protobuf', 'safetensors')
print('[install] pinned stack done')

In [ ]:
# === 2. GPU sanity ===
import subprocess
print('--- GPU ---')
subprocess.run(['nvidia-smi', '-L'], check=False)
import torch
print('CUDA OK?', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
# === 3. Verify attached Kaggle Models + suppress warning spam ===
import os, pathlib, shutil, warnings, logging

ACTOR_PATH  = '/kaggle/input/models/Microsoft/phi-3/pytorch/phi-3.5-mini-instruct/2'
CRITIC_PATH = '/kaggle/input/models/deepseek-ai/deepseek-r1-0528/transformers/deepseek-r1-0528-qwen3-8b/1'

def verify(path, label):
    p = pathlib.Path(path)
    if not p.exists():
        raise SystemExit(f'{label} not found at {path}. Attach the matching Kaggle Model.')
    if not (any(p.glob('*.safetensors')) or any(p.glob('*.bin'))):
        raise SystemExit(f'{label} found at {path} but no weight files inside.')
    print(f'{label}: OK -> {path}')

verify(ACTOR_PATH,  'actor  (Phi-3.5-mini-instruct)')
verify(CRITIC_PATH, 'critic (DeepSeek-R1-0528-Qwen3-8B)')

os.environ['HF_HOME']              = '/tmp/hf-cache'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/tmp/hf-cache'
os.environ['TRANSFORMERS_CACHE']   = '/tmp/hf-cache'
pathlib.Path('/tmp/hf-cache').mkdir(parents=True, exist_ok=True)

modules_dir = pathlib.Path('/tmp/hf-cache/modules')
if modules_dir.exists():
    shutil.rmtree(modules_dir, ignore_errors=True)
    print('cleared cached custom modeling code at', modules_dir)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN attached from Kaggle Secrets')
except Exception:
    print('No HF_TOKEN -- training runs fully offline (that is fine).')

for pat in ('.*Caching is incompatible with gradient checkpointing.*',
            '.*None of the inputs have requires_grad=True.*',
            '.*use_reentrant parameter should be passed explicitly.*',
            ".*AccumulateGrad node's stream does not match.*"):
    warnings.filterwarnings('ignore', message=pat)

class _PhiFilter(logging.Filter):
    def filter(self, r):
        return 'Caching is incompatible' not in r.getMessage()
for n in ('transformers', 'transformers.models.phi3.modeling_phi3',
          'torch.utils.checkpoint'):
    logging.getLogger(n).addFilter(_PhiFilter())
print('warning filters installed')

In [ ]:
# === 4. Clone the repo (fresh every run, prints commit hash) ===
import os, subprocess, pathlib, shutil
WORK = '/kaggle/working/incident-commander'
os.chdir('/kaggle/working')
if pathlib.Path(WORK).exists():
    shutil.rmtree(WORK, ignore_errors=True)
subprocess.run(['git', 'clone', '--depth', '1',
                'https://github.com/r1cksync/meta-rl-hack.git', WORK],
               check=True)
os.chdir(WORK)
subprocess.run(['git', '-C', WORK, 'log', '-1', '--oneline'], check=False)
print('cwd =', os.getcwd())

In [ ]:
# === 5. Configure run (shard 1 / 3) ===
import os
os.environ['INCIDENT_COMMANDER_MOCK'] = 'true'
os.environ['IC_ACTOR_MODEL']     = ACTOR_PATH
os.environ['IC_CRITIC_PROVIDER'] = 'local'
os.environ['IC_CRITIC_MODEL']    = CRITIC_PATH
os.environ['IC_TASK_MODE']       = 'all'
os.environ['IC_TASK_SHARDS']     = '3'
os.environ['IC_TASK_SHARD']      = '0'
os.environ['IC_TOTAL_UPDATES']   = '60'
os.environ['IC_ROLLOUTS']        = '3'
os.environ['IC_MAX_STEPS']       = '12'
os.environ['IC_CKPT_EVERY']      = '15'
os.environ['IC_RUN_NAME']        = 'kaggle1'
print('actor :', os.environ['IC_ACTOR_MODEL'])
print('critic:', os.environ['IC_CRITIC_MODEL'])
print('shard :', os.environ['IC_TASK_SHARD'], '/', os.environ['IC_TASK_SHARDS'])

In [ ]:
# === 6. Train ===
import subprocess, sys
result = subprocess.run([sys.executable, 'scripts/run_training.py'], check=False)
print('exit code:', result.returncode)

In [ ]:
# === 7. Package outputs for download ===
import shutil, glob, pathlib
LOGS = pathlib.Path('colab/logs')
finals = sorted(LOGS.glob('adapter_kaggle1_final'))
ckpts  = sorted(LOGS.glob('adapter_kaggle1_u*'))
keep = (finals or ckpts)
assert keep, 'No adapter directories found -- check the training cell output.'
src = keep[-1]
print('packaging', src)
dst = pathlib.Path('/kaggle/working/adapter_kaggle1.zip')
shutil.make_archive(str(dst.with_suffix('')), 'zip', root_dir=src)
print('zipped to', dst, 'size:', dst.stat().st_size, 'bytes')
for j in glob.glob('colab/logs/training_kaggle1*.json'):
    shutil.copy(j, '/kaggle/working/')
print('files in /kaggle/working/:')
for f in sorted(pathlib.Path('/kaggle/working/').iterdir()):
    if f.name == 'hf-cache':
        continue
    print(' ', f.name, f.stat().st_size if f.is_file() else '<dir>')

## Done

Download `adapter_kaggle1.zip` from the **Output** tab on the right.
Run shard 2 and shard 3 in parallel browser tabs, then on your laptop:

```powershell
python scripts/merge_lora_adapters.py `
    --inputs ./adapter_kaggle1 ./adapter_kaggle2 ./adapter_kaggle3 `
    --output ./adapter_merged
```

The merged adapter loads with the standard `peft` API on top of `microsoft/Phi-3.5-mini-instruct`.